# Module 04 · Chunking & the Corpus

### A hands-on, build-it-yourself module for advanced high school researchers

One sentence can defeat a whole-document embedding — once you split a 2 000-word
file into overlapping 500-character windows, each window gets its own vector and
your searches become dramatically more precise. In this module you will chunk the
**8-file metals-market corpus**, embed every chunk with a local
`sentence-transformers` model (no API key), index the vectors in Qdrant
`:memory:`, and run your first real retrieval queries. This is Module 04 of a
twelve-part track that ends in a full **Agentic RAG Evaluation** capstone.

## 📋 Summary: the one-paragraph version

Long documents are bad inputs for retrieval: a single embedding averages every
idea in the file, diluting specific facts into noise. **Chunking** solves this by
slicing each document into short, overlapping windows — each window becomes its
own vector, so a query about "LBMA auctions" lands near the right passage instead
of competing against everything else in a 2 000-word file. We use LangChain's
`RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=60)`, the exact
setting the capstone uses, with a local `all-MiniLM-L6-v2` embedder that requires
no API key.

## 🗺️ What you will build, step by step

| Step | What you do | Key tool |
| ---: | --- | --- |
| 0 | Set up the environment | `uv`, `jupyterlab` |
| 1 | Inspect the corpus — count files, preview one document | `pathlib.Path` |
| 2 | Chunk all 8 files with RecursiveCharacterTextSplitter | `langchain-text-splitters` |
| 3 | Embed all chunks with the local sentence-transformers model | `sentence-transformers` |
| 4 | Build a Qdrant `:memory:` collection and upsert all vectors | `qdrant-client` |
| 5 | Run sample queries and inspect top-k results | `qdrant_client.QdrantClient` |
| 6 | Experiment: change chunk_size and observe the effect | your own exploration |

### 🎓 What you will *learn* (the concepts)

- **Why chunking matters**: how whole-document embeddings dilute retrieval quality.
- **RecursiveCharacterTextSplitter**: what `chunk_size` and `chunk_overlap` do, and
  how the splitter chooses where to break.
- **The 8-file metals corpus**: the real knowledge base all later modules evaluate.
- **End-to-end keyless pipeline**: read → split → embed → index → query without
  any cloud API.

### ✅ Prerequisites

- Modules 01–03 of this track (you understand embeddings and cosine search).
- Comfort reading basic Python.
- Curiosity.

---
# Step 0 · Setup

## 0.1 Install the libraries

The exact dependency list lives in **`pyproject.toml`** next to this notebook,
so the environment is **reproducible**. Install everything with one command,
**from this module's folder**, using [`uv`](https://docs.astral.sh/uv/):

```bash
uv sync            # reads pyproject.toml, creates .venv/, installs everything
uv run jupyter lab # launch Jupyter inside that environment
```

When the notebook opens, pick the kernel **`Python 3 (ipykernel)`** (top-right
kernel picker). That is the interpreter from `.venv`, so every `import` below
resolves against what `uv sync` installed.

> **No API keys needed.** This module is entirely keyless — the local
> `sentence-transformers` model downloads from HuggingFace once (~90 MB) and
> then works offline. Qdrant runs fully in-process.

---
# Step 1 · Inspect the corpus

The `corpus/` folder next to this notebook contains 8 Markdown files — the
same knowledge base the capstone uses to answer questions about precious-metals
markets. Let's load them and take a look before splitting anything.

In [1]:
from pathlib import Path

CORPUS_DIR = Path("corpus")
corpus_paths = sorted(CORPUS_DIR.glob("*.md"))

print(f"Found {len(corpus_paths)} corpus files:\n")
for p in corpus_paths:
    word_count = len(p.read_text().split())
    print(f"  {p.name:<45} {word_count:>5} words")

Found 8 corpus files:

  01_spot_price_and_quotes.md                     322 words
  02_gold_fundamentals.md                         311 words
  03_silver_and_industrial_demand.md              301 words
  04_platinum_palladium.md                        277 words
  05_investment_vehicles.md                       314 words
  06_drivers_and_macro.md                         278 words
  07_risk_and_portfolio.md                        286 words
  08_futures_and_derivatives.md                   311 words


Preview the first 500 characters of one file so we know what the text looks like
before splitting.

In [2]:
sample_path = corpus_paths[0]
sample_text = sample_path.read_text()

print(f"=== {sample_path.name} (first 500 chars) ===\n")
print(sample_text[:500])
print("\n[…truncated…]")

=== 01_spot_price_and_quotes.md (first 500 chars) ===

# How Precious Metal Prices Are Quoted

The **spot price** of a precious metal is the price for immediate delivery of one unit of the metal, right now, in the wholesale market. For gold, silver, platinum, and palladium the standard unit is the **troy ounce (toz)**, which is about 31.1035 grams, slightly heavier than the ordinary avoirdupois ounce of 28.35 grams. Because the unit matters, a price feed will always state both a currency and a unit, for example "USD per troy ounce".

A live quote is

[…truncated…]


---
# Step 2 · Chunk with RecursiveCharacterTextSplitter

`RecursiveCharacterTextSplitter` breaks text on the most natural boundary it
can find — double newlines first (paragraph breaks), then single newlines, then
spaces, then characters as a last resort. This keeps chunks coherent: they
almost always end at a sentence boundary.

Two parameters control the split:

- **`chunk_size=500`** — maximum characters per chunk (≈ one dense paragraph).
- **`chunk_overlap=60`** — characters shared between consecutive chunks; prevents
  a key sentence from being cut in half between two windows.

> ⚠ **Caution — size matters a lot.**
> `chunk_size=50` gives fragments with no context; `chunk_size=2000` buries
> specific facts in a whole-page average. 500 / 60 is the capstone's tuned
> default. Always inspect a few chunks before committing to a setting.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=60,
)

# Load all corpus files, then split each one
raw_docs = [
    {"source": p.name, "page_content": p.read_text()}
    for p in corpus_paths
]

# chunks is a list of (source_filename, chunk_text) tuples
chunks: list[tuple[str, str]] = [
    (doc["source"], piece)
    for doc in raw_docs
    for piece in splitter.split_text(doc["page_content"])
]

print(f"Total chunks: {len(chunks)}")
print(f"Average chunk length: {sum(len(t) for _, t in chunks) / len(chunks):.0f} chars")

/Users/pmui/SynologyDrive/research/2026/summer2026/asdrp-agent-research/tutorials/04_chunking_corpus/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks: 43
Average chunk length: 354 chars


Let's peek at a few chunks to confirm they look like coherent passages.

In [4]:
import random

random.seed(42)
sample_indices = random.sample(range(len(chunks)), k=3)

for idx in sample_indices:
    source, text = chunks[idx]
    print(f"--- Chunk #{idx}  (source: {source}) ---")
    print(text)
    print()

--- Chunk #40  (source: 08_futures_and_derivatives.md) ---
Futures for different delivery months form a **curve**. When far-dated contracts cost more than near-dated ones, the market is in **contango**, which is the normal state for gold and reflects storage and financing costs. When near-dated contracts cost more than far-dated ones, the market is in **backwardation**, which can signal tight near-term supply. The shape of the curve affects the return of products that must "roll" from one contract to the next, a cost that can quietly erode returns in

--- Chunk #7  (source: 02_gold_fundamentals.md) ---
**Central banks** deserve special attention. They hold gold as part of their foreign reserves, and when many central banks buy at once, as happened in several recent years, it provides a strong source of demand that is fairly insensitive to price. Conversely, large official sales can weigh on the market.

--- Chunk #1  (source: 01_spot_price_and_quotes.md) ---
A live quote is not a singl

Notice that consecutive chunks share their last ~60 characters with the next
chunk's first ~60 characters (the `chunk_overlap`). This overlap is what
prevents a sentence straddling a boundary from being invisible to retrieval.

In [5]:
# Demonstrate overlap between two consecutive chunks from the same document
first_file_chunks = [t for s, t in chunks if s == corpus_paths[0].name]

if len(first_file_chunks) >= 2:
    c0 = first_file_chunks[0]
    c1 = first_file_chunks[1]
    print(f"Chunk 0 ends with:    …{c0[-70:]!r}")
    print(f"Chunk 1 starts with:  {c1[:70]!r}…")
    print()
    # Compute the actual overlap
    overlap_len = 0
    for n in range(min(len(c0), len(c1)), 0, -1):
        if c0[-n:] == c1[:n]:
            overlap_len = n
            break
    print(f"Exact overlap found: {overlap_len} characters")

Chunk 0 ends with:    …'ys state both a currency and a unit, for example "USD per troy ounce".'
Chunk 1 starts with:  'A live quote is not a single number. Dealers publish a **bid** (the pr'…

Exact overlap found: 0 characters


---
# Step 3 · Embed all chunks locally

We use `all-MiniLM-L6-v2` from `sentence-transformers` — the same 384-dimension
model from Modules 02 and 03. It downloads from HuggingFace the first time
(~90 MB) and then runs entirely offline.

Encoding all chunks at once in a single `.encode()` call is much faster than
encoding them one by one — the library batches them internally.

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_NAME = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(MODEL_NAME)

texts = [text for _, text in chunks]
vectors: np.ndarray = embedder.encode(texts, show_progress_bar=True)

print(f"\nEmbedded {len(vectors)} chunks")
print(f"Vector shape: {vectors.shape}  (chunks × dimensions)")
print(f"Each vector is {vectors.shape[1]}-dimensional (all-MiniLM-L6-v2 default)")

Batches: 100%|██████████| 2/2 [00:00<00:00,  3.11it/s]


Embedded 43 chunks
Vector shape: (43, 384)  (chunks × dimensions)
Each vector is 384-dimensional (all-MiniLM-L6-v2 default)


---
# Step 4 · Index in Qdrant `:memory:`

Qdrant is a vector database. When you pass `":memory:"` as the location it
runs entirely in RAM — no server to start, no files on disk. Perfect for
learning and experimentation.

We store each vector alongside a small **payload**: the chunk text and its
source filename. Payloads let us inspect which document a result came from.

In [7]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

COLLECTION = "metals_kb"
VECTOR_SIZE = vectors.shape[1]   # 384

client = QdrantClient(":memory:")
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
)

points = [
    PointStruct(
        id=i,
        vector=v.tolist(),
        payload={"text": text, "source": source},
    )
    for i, ((source, text), v) in enumerate(zip(chunks, vectors))
]
client.upsert(collection_name=COLLECTION, points=points)

info = client.get_collection(COLLECTION)
print(f"Collection '{COLLECTION}' ready.")
print(f"  Points indexed: {info.points_count}")
print(f"  Vector size:    {info.config.params.vectors.size}")
print(f"  Distance:       {info.config.params.vectors.distance}")

Collection 'metals_kb' ready.
  Points indexed: 43
  Vector size:    384
  Distance:       Cosine


---
# Step 5 · Retrieve top-k chunks for a query

Retrieval works the same way as in Module 03: encode the query into a vector,
then ask Qdrant for the nearest `k` neighbours by cosine similarity.

The difference now is that each result is a focused *chunk* — roughly one
paragraph — rather than a whole document. That precision is exactly what lets a
later LLM answer questions accurately.

In [11]:
def retrieve(query: str, k: int = 5) -> list[dict]:
    """Return the top-k matching chunks with source and score."""
    q_vec = embedder.encode(query).tolist()
    hits = client.query_points(
        collection_name=COLLECTION,
        query=q_vec,
        limit=k,
        with_payload=True,
    ).points
    return [
        {
            "score": round(hit.score, 4),
            "source": hit.payload["source"],
            "text": hit.payload["text"],
        }
        for hit in hits
    ]

In [12]:
# --- Query 1: LBMA auction mechanics ---
query_1 = "How is the LBMA gold price set?"
results_1 = retrieve(query_1, k=5)

print(f"Query: {query_1!r}\n")
for i, r in enumerate(results_1, 1):
    print(f"[{i}] score={r['score']}  source={r['source']}")
    print(f"    {r['text'][:150].replace(chr(10), ' ')}…")
    print()

Query: 'How is the LBMA gold price set?'

[1] score=0.6477  source=01_spot_price_and_quotes.md
    The **London Bullion Market Association (LBMA)** publishes twice-daily reference prices known historically as the **London Fix**, set during electroni…

[2] score=0.5932  source=01_spot_price_and_quotes.md
    # How Precious Metal Prices Are Quoted  The **spot price** of a precious metal is the price for immediate delivery of one unit of the metal, right now…

[3] score=0.5589  source=01_spot_price_and_quotes.md
    A few practical points follow from all of this. First, two providers can show slightly different "gold prices" because they draw on different exchange…

[4] score=0.5311  source=02_gold_fundamentals.md
    **Central banks** deserve special attention. They hold gold as part of their foreign reserves, and when many central banks buy at once, as happened in…

[5] score=0.531  source=06_drivers_and_macro.md
    Finally, **supply and demand specifics** matter more for the industria

In [13]:
# --- Query 2: Silver industrial demand ---
query_2 = "What role does silver play in solar panels?"
results_2 = retrieve(query_2, k=5)

print(f"Query: {query_2!r}\n")
for i, r in enumerate(results_2, 1):
    print(f"[{i}] score={r['score']}  source={r['source']}")
    print(f"    {r['text'][:150].replace(chr(10), ' ')}…")
    print()

Query: 'What role does silver play in solar panels?'

[1] score=0.5529  source=03_silver_and_industrial_demand.md
    Industrial uses include **solar photovoltaic cells**, electronics and electrical contacts, brazing alloys, and medical and antibacterial applications.…

[2] score=0.4801  source=03_silver_and_industrial_demand.md
    On the supply side, most silver is produced as a **by-product** of mining other metals such as copper, lead, zinc, and gold. This is important: when a…

[3] score=0.4531  source=03_silver_and_industrial_demand.md
    # Silver: Half Money, Half Industrial Metal  Silver behaves like two assets at once. Like gold, it is a **monetary and investment metal**, bought as b…

[4] score=0.4196  source=03_silver_and_industrial_demand.md
    Because silver is both an investment metal and an industrial input, its price is **more volatile** than gold's. In a strong economy with rising invest…

[5] score=0.3582  source=07_risk_and_portfolio.md
    A metal allocation carri

In [14]:
# --- Query 3: Portfolio / risk ---
query_3 = "How does gold help diversify an investment portfolio?"
results_3 = retrieve(query_3, k=5)

print(f"Query: {query_3!r}\n")
for i, r in enumerate(results_3, 1):
    print(f"[{i}] score={r['score']}  source={r['source']}")
    print(f"    {r['text'][:150].replace(chr(10), ' ')}…")
    print()

Query: 'How does gold help diversify an investment portfolio?'

[1] score=0.8227  source=07_risk_and_portfolio.md
    # Precious Metals in a Portfolio: Risk and Diversification  Investors hold precious metals mainly for **diversification**. The argument is that gold i…

[2] score=0.797  source=07_risk_and_portfolio.md
    The diversification benefit is strongest precisely when it is most useful: during severe equity sell-offs and crises, gold has often risen or held ste…

[3] score=0.6144  source=02_gold_fundamentals.md
    Gold's most famous property is its reputation as a **safe-haven asset**. During financial crises, sharp stock-market declines, or geopolitical shocks,…

[4] score=0.5694  source=06_drivers_and_macro.md
    The most important driver for gold is **real interest rates**, the interest rate minus expected inflation. Gold produces no income, so its appeal rise…

[5] score=0.5376  source=02_gold_fundamentals.md
    # Gold: Supply, Demand, and the Safe-Haven Role  Gold is u

---
# Step 6 · Experiment — how does chunk_size change results?

Let's rebuild the index with a *larger* chunk size and compare. This is the
kind of quick experiment you would run before committing to a chunking strategy
in a real project.

> ⚠ **Caution — there is no universal best chunk size.**
> The right value depends on your embedding model's ideal input length, your
> documents' structure, and your retrieval task. Always run an evaluation (see
> Module 07) rather than trusting intuition alone.

In [15]:
def build_index(chunk_size: int, chunk_overlap: int) -> QdrantClient:
    """Build a fresh in-memory Qdrant index with the given split parameters."""
    sp = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    ch = [
        (doc["source"], piece)
        for doc in raw_docs
        for piece in sp.split_text(doc["page_content"])
    ]
    vecs = embedder.encode([t for _, t in ch])
    cl = QdrantClient(":memory:")
    cl.create_collection(
        "exp",
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )
    cl.upsert("exp", points=[
        PointStruct(id=i, vector=v.tolist(),
                    payload={"text": t, "source": s})
        for i, ((s, t), v) in enumerate(zip(ch, vecs))
    ])
    return cl, len(ch)


print("Rebuilding index with chunk_size=1000, chunk_overlap=100 …")
client_large, n_large = build_index(1000, 100)
print(f"  Large chunks: {n_large} total (was {len(chunks)} at 500/60)")

Rebuilding index with chunk_size=1000, chunk_overlap=100 …
  Large chunks: 22 total (was 43 at 500/60)


In [ ]:
# Compare retrieval for the same query under both chunk sizes
test_query = "How is the LBMA gold price set?"
q_vec = embedder.encode(test_query).tolist()

hits_default = client.search(COLLECTION, query_vector=q_vec, limit=3, with_payload=True)
hits_large   = client_large.search("exp",     query_vector=q_vec, limit=3, with_payload=True)

print(f"Query: {test_query!r}\n")
print("=== Default (chunk_size=500) ===")
for h in hits_default:
    print(f"  score={h.score:.4f}  len={len(h.payload['text'])} chars")
    print(f"  {h.payload['text'][:120].replace(chr(10), ' ')}…\n")

print("=== Large (chunk_size=1000) ===")
for h in hits_large:
    print(f"  score={h.score:.4f}  len={len(h.payload['text'])} chars")
    print(f"  {h.payload['text'][:120].replace(chr(10), ' ')}…\n")

Larger chunks tend to produce *lower* cosine scores because the embedding
averages more content — the query signal is diluted. Smaller chunks have
higher scores but may lack the surrounding context an LLM needs to give
a complete answer. The 500 / 60 setting balances both concerns.

---
# Recap & what's next

You built a complete keyless retrieval pipeline over real documents:

1. **Loaded** 8 metals-market Markdown files with a `Path` glob.
2. **Chunked** them with `RecursiveCharacterTextSplitter(500, 60)` — natural
   paragraph-level windows with 60-character overlap.
3. **Embedded** every chunk locally with `all-MiniLM-L6-v2` (384-dim, no API key).
4. **Indexed** the vectors in Qdrant `:memory:` with source payloads.
5. **Retrieved** focused passages for three different queries.
6. **Experimented** with a larger chunk size to see precision vs. context trade-offs.

**Next module (05 — Stack migration + first real RAG):** you will swap the local
`sentence-transformers` embedder for cloud Ollama via LiteLLM, load your
`OLLAMA_API_KEY` from the shared `tutorials/.env`, and generate the first full
RAG answer — query → retrieve chunks → stuff prompt → LLM response.